# 01. Exploración Transaccional: Detección de Fraude en PaySim y Topología de Redes

## 1. Contexto, Delimitación Conceptual y Objetivos

### Naturaleza de los Datos y Advertencia Metodológica
Este estudio utiliza *PaySim*, un simulador multi-agente basado en registros agregados de dinero móvil de un operador financiero en África. Al ser un entorno sintético, los comportamientos de fraude están determinados por reglas probabilísticas y heurísticas preprogramadas. Comprender estas reglas es indispensable para evitar caer en trampas de sobreajuste o en el aprendizaje de atajos (*shortcut learning*).

### Distinción Conceptual: Fraude Transaccional vs. Lavado de Activos (AML)
Es fundamental precisar el alcance analítico:
- Las etiquetas de fraude (`isFraud`) en PaySim modelan primordialmente *ataques de sustracción y drenado no autorizado de fondos* (toma de control de cuentas y colusión de agentes).
- No modelan el ciclo completo de Lavado de Dinero (AML), el cual involucra fases macroestructurales de *colocación, estratificación (layering) e integración*.
- En este proyecto, nos enfocamos en detectar la fase operativa de desvío y liquidación rápida, evaluando cómo las señales contables y topológicas interactúan para delatar estas operaciones.

### Delimitación Operativa: ¿Por qué filtramos solo `TRANSFER` y `CASH_OUT`?
El dataset completo contiene 5 tipos de transacciones (`CASH-IN`, `CASH-OUT`, `DEBIT`, `PAYMENT` y `TRANSFER`), pero filtramos estrictamente **`TRANSFER`** y **`CASH_OUT`** por dos razones estructurales del simulador:
1. **Vector de Fraude Exclusivo:** El motor sintético inyecta el 100% de los fraudes únicamente en estas dos tipologías:
   - El atacante compromete fondos y los desvía a una cuenta puente vía `TRANSFER`.
   - Luego, retira o liquida ese saldo hacia el exterior vía `CASH_OUT`.
2. **Exclusión de Ruido:** Los flujos de `PAYMENT` (pagos en comercios), `CASH-IN` (depósitos) y `DEBIT` (débitos directos) registran cero instancias de fraude en el simulador. Filtrarlos elimina millones de registros irrelevantes y previene distorsiones artificiales en la matriz de adyacencia de la red.

### Pregunta Central de Investigación
*¿Es viable identificar transacciones fraudulentas basándonos en dinámica contable de saldos y topología de red sin que el modelo dependa exclusivamente de los sesgos sintéticos del simulador?*

### Hoja de Ruta de la Suite Experimental
1. *Notebook 01 (Este cuaderno):* Formulación de hipótesis, delimitación operativa, detección del atajo sintético de vaciado y extracción exploratoria de métricas de red.
2. *Notebook 02:* Análisis de granularidad del caudal (`ratio_vaciado`), auditoría de recurrencia de pares (`nameOrig`, `nameDest`) y sellado de características.
3. *Notebook 03:* Diseño experimental libre de fuga temporal, estudio formal de ablación (Modelos A, B y C) y atribución cuantitativa de importancia con SHAP.

In [7]:
import duckdb
import networkx as nx
import pandas as pd

print("Leyendo y filtrando el CSV original...")
conn = duckdb.connect()

# Agregamos ../ para retroceder a la raíz del proyecto
df = conn.execute("""
    SELECT * 
    FROM read_csv_auto('../data/raw/paysim.csv')
    WHERE type IN ('TRANSFER', 'CASH_OUT')
""").df()

print(f"Transacciones listas para analizar: {len(df)}")
display(df.head())

Leyendo y filtrando el CSV original...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Transacciones listas para analizar: 2770409


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0
1,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0
2,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0
3,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0
4,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0


## 2. Dinámica Contable: El Atajo Sintético de Vaciado (*Shortcut Learning*)

En un motor transaccional, una transferencia no autorizada suele ir acompañada de una anomalía en los saldos. Al analizar el comportamiento del agente defraudador en PaySim, evaluamos la relación entre el monto transferido (`amount`) y el saldo inicial previo a la operación (`oldbalanceOrg`).

Definimos el sensor binario de vaciado exacto:
$$\text{flag\_vaciado} = \mathbb{I}(\text{amount} == \text{oldbalanceOrg})$$

**Advertencia de Auditoría:**  
Si la gran mayoría de los fraudes en la simulación cumplen esta igualdad exacta, nos enfrentamos a un **atajo sintético del generador de datos**. Un modelo de Machine Learning que use esta señal alcanzará un desempeño artificialmente perfecto en el simulador, pero fallará en un entorno bancario real de producción donde los atacantes diversifican los montos o realizan retiros parciales para no activar reglas estáticas de cumplimiento.

In [8]:
# Filtramos solo los casos de fraude comprobado
df_fraud = df[df['isFraud'] == 1].copy()

# Creamos nuestro primer sensor lógico (flag_vaciado)
df_fraud['flag_vaciado'] = (df_fraud['amount'] == df_fraud['oldbalanceOrg']).astype(int)

vaciados = df_fraud['flag_vaciado'].sum()
total_fraudes = len(df_fraud)

print(f"De {total_fraudes} fraudes, {vaciados} vaciaron la cuenta origen a cero.")
display(df_fraud[['amount', 'oldbalanceOrg', 'newbalanceOrig', 'flag_vaciado']].head())

De 8213 fraudes, 8034 vaciaron la cuenta origen a cero.


,amount,oldbalanceOrg,newbalanceOrig,flag_vaciado
0,181.0,181.0,0.0,1
1,181.0,181.0,0.0,1
123,2806.0,2806.0,0.0,1
124,2806.0,2806.0,0.0,1
192,20128.0,20128.0,0.0,1


## 3. Topología de Red: Detección de Estructuras Concentradoras y Dispersoras

El 97.8% de los fraudes en PaySim cumplen la regla exacta de vaciado a cero ($8,034$ de $8,213$). Sin embargo, para construir defensas robustas frente al 2.2% restante y ante tipologías reales, debemos recurrir a la estructura del grafo transaccional.

Modelamos las transacciones como un grafo dirigido $G = (V, E)$, donde las cuentas son nodos y las operaciones son aristas dirigidas:
- **In-Degree ($\text{fan-in}$):** Cantidad de orígenes únicos que transfieren fondos hacia una cuenta destino común. Permite mapear puntos de recolección de fondos (posibles cuentas mula o nodos receptores de embudo).
- **Out-Degree ($\text{fan-out}$):** Cantidad de destinos únicos a los que una cuenta transfiere. Identifica dispersión masiva de fondos.

---

###  Nota de Auditoría sobre Fuga Temporal (Data Leakage)
> En esta celda exploratoria, `in_degree` y `out_degree` se calculan agrupando sobre **la totalidad de la muestra histórica**.  
> **Limitación metodológica:** Este cálculo global utiliza información del futuro relativo respecto a los primeros pasos temporales (`step`).  
> **Solución aplicada en Notebook 03:** En la fase de modelado predictivo, este cálculo global se descarta formalmente y se sustituye por `in_degree_hist`, calculado de manera estrictamente causal y acumulativa exclusivamente sobre la partición de entrenamiento (`step <= 475`), asignando grado cero a cuentas nuevas para garantizar un flujo sin fuga temporal.

In [9]:
# Contamos de cuántas cuentas distintas recibe dinero un destino (in-degree)
fan_in = df.groupby('nameDest')['nameOrig'].nunique().reset_index(name='in_degree')

# Contamos a cuántas cuentas distintas envía dinero un origen (out-degree)
fan_out = df.groupby('nameOrig')['nameDest'].nunique().reset_index(name='out_degree')

print("Top 5 cuentas destino (Posibles mulas colectoras - alto in-degree):")
display(fan_in.sort_values('in_degree', ascending=False).head())

print("\nTop 5 cuentas origen (Posibles dispersores - alto out-degree):")
display(fan_out.sort_values('out_degree', ascending=False).head())

Top 5 cuentas destino (Posibles mulas colectoras - alto in-degree):


,nameDest,in_degree
75478,C1286084959,75
421126,C665576141,68
95247,C1360767589,68
503591,C97730845,67
311527,C248609774,64



Top 5 cuentas origen (Posibles dispersores - alto out-degree):


,nameOrig,out_degree
1291341,C1902386530,3
1572805,C2098525306,3
2372816,C724452879,3
69019,C1048374464,2
422324,C129588043,2


## 4. Integración de Métricas de Red al Conjunto de Datos

Unimos las métricas topológicas al conjunto general de `TRANSFER` y `CASH_OUT` para contrastar las distribuciones de grado entre transacciones legítimas y fraudulentas.

In [4]:
# Unimos las features topológicas al dataset original
df = df.merge(fan_in, on='nameDest', how='left')
df = df.merge(fan_out, on='nameOrig', how='left')

# Llenamos con 0 los casos donde no hay conexiones adicionales
df['in_degree'] = df['in_degree'].fillna(0)
df['out_degree'] = df['out_degree'].fillna(0)

# Comparamos el comportamiento de red entre operaciones normales y lavado de dinero
analisis_red = df.groupby('isFraud')[['in_degree', 'out_degree']].mean().round(2)

print("Promedio de conexiones (Legítimo vs Fraude):")
display(analisis_red)

Promedio de conexiones (Legítimo vs Fraude):


,in_degree,out_degree
isFraud,,
0,11.6,1.0
1,5.7,1.0


## 5. Intersección de Sensores: Desacoplando Comercios de Cuentas Sospechosas

### El Falso Positivo Estructural
Un análisis superficial de grafos asumiría que a mayor grado de entrada (`in_degree`), mayor probabilidad de lavado de dinero. Sin embargo, los datos empíricos revelan lo contrario:
- Las transacciones legítimas tienen un `in_degree` promedio significativamente más alto ($\approx 11.60$) porque reflejan pasarelas de pago, recaudadores y comercios que reciben transferencias legítimas de cientos de clientes distintos.
- Los fraudes operan en estructuras mucho más discretas y acotadas (promedio $\approx 5.71$).

Por lo tanto, la topología por sí sola es un pésimo clasificador. La señal emerge únicamente al evaluar la **intersección** entre la física del saldo (el vaciado del origen) y la escala de la red del receptor.

In [5]:
# Aplicamos el sensor de vaciado a TODO el dataset (no solo a los fraudes como antes)
df['flag_vaciado'] = (df['amount'] == df['oldbalanceOrg']).astype(int)

# Agrupamos por si es fraude y si hubo vaciado, viendo el comportamiento de la red
analisis_combinado = df.groupby(['isFraud', 'flag_vaciado'])['in_degree'].agg(['count', 'mean']).round(2)
analisis_combinado.columns = ['cantidad_transacciones', 'in_degree_promedio']

print("Intersección de Sensores: Vaciado x Topología")
display(analisis_combinado)

Intersección de Sensores: Vaciado x Topología


cantidad_transacciones  in_degree_promedio
isFraud flag_vaciado                                            
0       0                            2762196               11.60
1       0                                179                5.36
        1                               8034                5.71

## 6. Conclusiones de la Fase Exploratoria y Pase de Posta a Notebook 02

### Hallazgos Clave de la Auditoría:
1. **Dependencia Heurística del Simulador:** El 97.8% de los fraudes en PaySim ($8,034$ casos) están atados a la regla fija de vaciado exacto (`amount == oldbalanceOrg`). No obstante, existe un grupo residual de **179 fraudes atípicos** que no vacían la cuenta a cero o presentan montos divergentes.
2. **Topología Polarizada:** El grado de entrada no es suficiente como señal univariada debido a la presencia de cuentas comerciales legítimas con alto tráfico. Debe ser evaluado en conjunto con la variación del saldo.
3. **Identificación de Fuga Temporal:** Se detecta que el cálculo estático de grados sobre toda la serie genera fuga de información hacia el pasado, lo que motiva la formulación de grados históricos en el Notebook 03.

### Preguntas Abiertas que Aborda el Notebook 02:
- ¿Es el drenado de fondos estrictamente binario, o un ratio de caudal continuo ($\text{amount} / \text{oldbalanceOrg}$) captura mejor el comportamiento de los 179 casos atípicos?
- ¿Existe recurrencia temporal entre pares idénticos $(\text{nameOrig}, \text{nameDest})$ en las tipologías de dispersión?

Exportamos el archivo `paysim_features.parquet` para continuar la ingeniería y validación de sensores en el Notebook 02.

In [6]:
import os

# Aseguramos que la carpeta exista
os.makedirs('../data/processed', exist_ok=True)

# Guardamos el dataset enriquecido en formato Parquet
ruta_exportacion = '../data/processed/paysim_features.parquet'
df.to_parquet(ruta_exportacion, index=False)

print(f"Dataset con features de red guardado exitosamente en: {ruta_exportacion}")

Dataset con features de red guardado exitosamente en: ../data/processed/paysim_features.parquet
